# Phase 3: Data Preparation

**CRISP-DM Phase Description:**  
This phase covers all activities to construct the final dataset from the initial raw data. Data preparation tasks are likely to be performed multiple times, and not in any prescribed order. This is typically the longest and most time-consuming phase of the CRISP-DM lifecycle.

---

In [1]:
# Standard library imports for this phase
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
%matplotlib inline

In [2]:
# Load the dataset from Phase 2 (update the path as needed)
DATA_PATH = '../data/Malicious_URL_v3.csv'

df = pd.read_csv(DATA_PATH)
print(f"Loaded dataset: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

Loaded dataset: 49750 rows x 3 columns


,Unnamed: 0,url,type
0,0,br-icloud.com.br,phishing
1,1,mp3raid.com/music/krizz_kaliko.html,benign
2,2,bopsecrets.org/rexroth/cr/1.htm,benign
3,3,http://www.garage-pirenne.be/index.php?option=...,defacement
4,4,http://adventure-nicaragua.net/index.php?optio...,defacement


---
### Task 1: Select Data

Decide on the data to be used for analysis. Consider which columns (features) and rows (records) to include or exclude based on:

- **Relevance:** Does this feature contribute to the data mining goal?
- **Data Quality:** Is the quality of this feature sufficient (e.g., too many missing values)?
- **Technical Constraints:** Are there limitations on data volume or specific feature types?

**Output:** A rationale for inclusion/exclusion of data, and the resulting subset.

**Instructions:** Select the columns and rows relevant to your analysis goal. Document your reasoning.

In [8]:
# TODO: Select the relevant columns and rows for your analysis.
# Document your rationale for including or excluding specific features.

# Example:
columns_to_keep = ['url', 'type']
columns_to_drop = [df.columns[0]] 
drop_reason = {
    'df.columns[0]': 'Unique identifier, not predictive',
}

df_selected = df.drop(columns=columns_to_drop)
print(f"Shape after column selection: {df_selected.shape}")
df_selected.head()

Shape after column selection: (49750, 2)


,url,type
0,br-icloud.com.br,phishing
1,mp3raid.com/music/krizz_kaliko.html,benign
2,bopsecrets.org/rexroth/cr/1.htm,benign
3,http://www.garage-pirenne.be/index.php?option=...,defacement
4,http://adventure-nicaragua.net/index.php?optio...,defacement


In [ ]:
# Optional: Filter rows based on specific criteria
# Example: Remove rows where a critical field is missing or filter by a condition

# df_selected = df_selected[df_selected['some_column'].notna()]
# print(f"Shape after row selection: {df_selected.shape}")

---
### Task 2: Clean Data

Raise data quality to the level required by the selected analysis techniques. Cleaning activities include:

- **Handle Missing Values:** Impute missing values (mean, median, mode, forward/backward fill) or remove rows/columns with excessive missing data.
- **Correct Errors:** Fix inaccurate or corrupted data entries.
- **Remove Duplicates:** Eliminate exact or near-duplicate records.
- **Handle Outliers:** Decide how to treat extreme values (keep, cap, transform, or remove).

**Instructions:** Apply appropriate cleaning techniques to address the data quality issues identified in Phase 2, Task 4.

In [18]:
# TODO: Handle missing values.
# Choose an appropriate strategy for each column.

df_clean = df_selected.copy()

# 1. Handle missing values (minimal since dataset is mostly clean)
df_clean = df_clean.dropna(subset=['url', 'type'])

# 2. Remove empty URLs
df_clean = df_clean[df_clean['url'].str.strip() != ""]

# 3. Standardize target labels
df_clean['type'] = df_clean['type'].str.lower().str.strip()

# 4. Keep only valid classes
valid_classes = ['benign', 'phishing', 'malware', 'defacement']
df_clean = df_clean[df_clean['type'].isin(valid_classes)]

print(f"Shape after cleaning: {df_clean.shape}")
df_clean['type'].value_counts()
print(len(df_selected) - len(df_clean), "rows removed during cleaning.")



Shape after cleaning: (49750, 2)
0 rows removed during cleaning.


In [19]:
# TODO: Remove duplicate records.

before = len(df_clean)
df_clean = df_clean.drop_duplicates()
after = len(df_clean)
print(f"Removed {before - after} duplicate rows. Remaining: {after} rows.")



Removed 9311 duplicate rows. Remaining: 40439 rows.


In [ ]:
# TODO: Handle outliers.
# Choose a strategy: capping (winsorizing), removing, or transforming.

# Example: Cap outliers using the IQR method
# def cap_outliers_iqr(dataframe, column):
#     Q1 = dataframe[column].quantile(0.25)
#     Q3 = dataframe[column].quantile(0.75)
#     IQR = Q3 - Q1
#     lower_bound = Q1 - 1.5 * IQR
#     upper_bound = Q3 + 1.5 * IQR
#     dataframe[column] = dataframe[column].clip(lower=lower_bound, upper=upper_bound)
#     return dataframe

# for col in numerical_cols:
#     df_clean = cap_outliers_iqr(df_clean, col)

In [20]:
print("=== Outlier Handling ===\n")

print("At this stage, the dataset does not contain engineered numerical features suitable for traditional outlier treatment methods.\n")

print("A simple proxy variable (URL length) was analyzed during the Data Understanding phase, revealing the presence of extreme values.\n")

print("However, in the context of cybersecurity:")
print("- Outliers may represent malicious behavior (e.g., obfuscated URLs)")
print("- Removing or capping these values could eliminate important signals\n")

print("Decision:")
print("No outlier removal or transformation was applied.\n")

print("Justification:")
print("Outliers are retained as they are likely to carry meaningful information for detecting phishing and malicious URLs.\n")

print("Conclusion:")
print("The dataset will be further processed during feature engineering, where more meaningful numerical features will be created.")

=== Outlier Handling ===

At this stage, the dataset does not contain engineered numerical features suitable for traditional outlier treatment methods.

A simple proxy variable (URL length) was analyzed during the Data Understanding phase, revealing the presence of extreme values.

However, in the context of cybersecurity:
- Outliers may represent malicious behavior (e.g., obfuscated URLs)
- Removing or capping these values could eliminate important signals

Decision:
No outlier removal or transformation was applied.

Justification:
Outliers are retained as they are likely to carry meaningful information for detecting phishing and malicious URLs.

Conclusion:
The dataset will be further processed during feature engineering, where more meaningful numerical features will be created.


---
### Task 3: Construct Data (Feature Engineering)

This task involves creating new attributes (features) derived from existing ones that may be more useful for modelling. Common techniques include:

- **Derived Attributes:** Create new features from existing ones (e.g., extracting `year`, `month`, `day` from a datetime column; computing `total_spend = price * quantity`).
- **Binning / Discretisation:** Convert continuous variables into categorical bins (e.g., age groups).
- **Encoding Categorical Variables:** Convert categorical features into numerical representations (e.g., one-hot encoding, label encoding).
- **Scaling / Normalisation:** Scale numerical features to a common range (e.g., Min-Max scaling, Standardisation).

**Instructions:** Create new features or transform existing ones to improve model performance.

In [ ]:
# TODO: Create derived attributes / new features.

# Example: Create a new feature from existing columns
# df_clean['new_feature'] = df_clean['feature_a'] / df_clean['feature_b']

# Example: Extract components from a datetime column
# df_clean['date_column'] = pd.to_datetime(df_clean['date_column'])
# df_clean['year'] = df_clean['date_column'].dt.year
# df_clean['month'] = df_clean['date_column'].dt.month
# df_clean['day_of_week'] = df_clean['date_column'].dt.dayofweek

In [ ]:
# TODO: Encode categorical variables.

# Example: One-hot encoding
# df_clean = pd.get_dummies(df_clean, columns=['categorical_col_1', 'categorical_col_2'], drop_first=True)

# Example: Label encoding for ordinal variables
# from sklearn.preprocessing import LabelEncoder
# le = LabelEncoder()
# df_clean['ordinal_col'] = le.fit_transform(df_clean['ordinal_col'])

In [ ]:
# TODO: Scale / normalise numerical features if required.

# from sklearn.preprocessing import StandardScaler, MinMaxScaler

# scaler = StandardScaler()  # or MinMaxScaler()
# cols_to_scale = ['feature_1', 'feature_2']
# df_clean[cols_to_scale] = scaler.fit_transform(df_clean[cols_to_scale])

---
### Task 4: Integrate Data

If your project uses multiple data sources, this task involves merging or combining them into a single, unified dataset. Activities include:

- **Merging Tables:** Join datasets on common keys (e.g., using `pd.merge()`).
- **Appending Records:** Concatenate datasets with the same structure (e.g., using `pd.concat()`).
- **Aggregation:** Summarise data at a different level of granularity.

**Instructions:** If using multiple data sources, merge or concatenate them below. If your project uses a single dataset, document that here and proceed to the next task.

In [ ]:
# TODO: Integrate data from multiple sources (if applicable).

# Example: Merge two DataFrames on a common key
# df_secondary = pd.read_csv('path/to/secondary_data.csv')
# df_integrated = pd.merge(df_clean, df_secondary, on='common_key', how='left')

# Example: Concatenate DataFrames with the same structure
# df_integrated = pd.concat([df_part1, df_part2], ignore_index=True)

# If using a single source, simply continue:
# df_integrated = df_clean.copy()
# print(f"Integrated dataset shape: {df_integrated.shape}")

In [ ]:
# Optional: Verify the integrated data
# df_integrated.head()

---
### Task 5: Format Data

This final preparation task ensures the data is in the correct format for the modelling tools. Activities include:

- **Data Type Conversions:** Ensure all columns have appropriate data types (e.g., numeric, datetime, categorical).
- **Column Reordering:** Arrange columns in a logical order (e.g., features first, target last).
- **Renaming:** Give columns clear, descriptive names.
- **Saving the Prepared Dataset:** Export the final, clean dataset for use in subsequent phases.

**Instructions:** Apply any final formatting changes and save the prepared dataset.

In [ ]:
# TODO: Apply final formatting — data types, column order, renaming.

# Example: Convert data types
# df_integrated['some_column'] = df_integrated['some_column'].astype('int')

# Example: Rename columns for clarity
# df_integrated = df_integrated.rename(columns={
#     'old_name': 'new_descriptive_name'
# })

# Example: Reorder columns (target column last)
# target_col = 'target'
# feature_cols = [col for col in df_integrated.columns if col != target_col]
# df_final = df_integrated[feature_cols + [target_col]]

In [ ]:
# TODO: Verify the final prepared dataset.

# print("=" * 60)
# print("FINAL PREPARED DATASET SUMMARY")
# print("=" * 60)
# print(f"Shape: {df_final.shape}")
# print(f"Missing values: {df_final.isnull().sum().sum()}")
# print(f"Duplicates: {df_final.duplicated().sum()}")
# print(f"\nColumn types:")
# print(df_final.dtypes)
# df_final.head()

In [ ]:
# TODO: Save the prepared dataset for use in Phase 4 (Modelling).

# OUTPUT_PATH = 'path/to/your/prepared_data.csv'
# df_final.to_csv(OUTPUT_PATH, index=False)
# print(f"Prepared dataset saved to: {OUTPUT_PATH}")